<a href="https://colab.research.google.com/github/jferreira04-cloud/Calculo_num-rico_lobosco/blob/main/animacao_eliminacao_gauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Animação interativa da Eliminação de Gauss (sistemas n × n)

A animação mostra, quadro a quadro, a eliminação de Gauss do material: matriz aumentada $[A\,|\,b]$, escolha do pivô, trocas de linhas, multiplicadores $m_{ik} = a_{ik}/a_{kk}$, a operação $R_i \leftarrow R_i - m_{ik}R_k$, a matriz triangular superior e a substituição regressiva.

* funciona em **dimensões diferentes** (de 1×1 até 8×8);
* **ao editar as equações, a animação é refeita na hora**;
* compara **com / sem pivoteamento parcial** e **frações exatas / ponto flutuante**.

**Como usar:** menu **Ambiente de execução → Executar tudo**. O painel aparece no fim da célula 3.

## 1. Núcleo: leitura do sistema, registro dos passos e desenho de cada quadro

In [1]:
# ==========================================================================
# Animação da Eliminação de Gauss (n×n) — núcleo: leitura, passos e desenho
# ==========================================================================
import io
import os
import re
import html
import random
from fractions import Fraction

import numpy as np
import matplotlib
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch
from matplotlib import animation
import ipywidgets as widgets
from IPython.display import display, HTML

matplotlib.rcParams['animation.embed_limit'] = 100  # MB (animação exportável em HTML)
N_MAX = 8  # maior dimensão aceita pela animação


# --------------------------------------------------------------------------
# 1) Leitura do sistema: equações ("2x + y - z = 8") ou matriz aumentada ("2 1 -1 | 8")
# --------------------------------------------------------------------------
class Sistema:
    """Sistema quadrado A x = b guardado com frações exatas."""

    def __init__(self, A, b, nomes):
        self.A = [[Fraction(v) for v in linha] for linha in A]
        self.b = [Fraction(v) for v in b]
        self.nomes = list(nomes)

    @property
    def n(self):
        return len(self.b)

    def chave(self):
        return (tuple(map(tuple, self.A)), tuple(self.b), tuple(self.nomes))

    def como_numpy(self):
        return np.array(self.A, dtype=float), np.array(self.b, dtype=float)

    def __repr__(self):
        eqs = [equacao_texto(l, bi, self.nomes) for l, bi in zip(self.A, self.b)]
        return f'Sistema {self.n}×{self.n}:\n  ' + '\n  '.join(eqs)


_TROCAS = {'−': '-', '–': '-', '—': '-', '·': '*', '×': '*', '⋅': '*', '÷': '/'}
_SUBSCRITOS = str.maketrans('₀₁₂₃₄₅₆₇₈₉', '0123456789')
_TOKEN = re.compile(r"""\s*(?:
    (?P<num>(?:\d+(?:[.,]\d*)?|[.,]\d+)(?:[eE][+-]?\d+)?) |
    (?P<var>[A-Za-z]\w*) |
    (?P<op>[-+*/()=])
)""", re.VERBOSE)


def _fracao(txt):
    return Fraction(txt.replace(',', '.'))


def _tokenizar(expr, nlin):
    toks, pos, expr = [], 0, expr.rstrip()
    while pos < len(expr):
        m = _TOKEN.match(expr, pos)
        if not m:
            raise ValueError(f'Linha {nlin}: não entendi o trecho "{expr[pos:].strip()[:15]}".')
        pos = m.end()
        if m.group('num') is not None:
            toks.append(('num', _fracao(m.group('num'))))
        elif m.group('var') is not None:
            toks.append(('var', m.group('var')))
        else:
            toks.append(('op', m.group('op')))
    return toks


class _Expressao:
    """Analisador de expressões lineares: devolve ({incógnita: coeficiente}, constante)."""

    def __init__(self, toks, nlin):
        self.t, self.i, self.nlin = toks, 0, nlin

    def erro(self, msg):
        raise ValueError(f'Linha {self.nlin}: {msg}')

    def olhar(self):
        return self.t[self.i] if self.i < len(self.t) else (None, None)

    def pegar(self):
        tok = self.olhar()
        self.i += 1
        return tok

    def completa(self):
        f = self.expr()
        if self.i < len(self.t):
            self.erro(f'símbolo inesperado "{self.t[self.i][1]}".')
        return f

    def expr(self):
        f = self.termo()
        while self.olhar() in (('op', '+'), ('op', '-')):
            sinal = 1 if self.pegar()[1] == '+' else -1
            g = self.termo()
            coefs = dict(f[0])
            for v, a in g[0].items():
                coefs[v] = coefs.get(v, 0) + sinal * a
            f = (coefs, f[1] + sinal * g[1])
        return f

    def termo(self):
        f = self.fator()
        while True:
            tipo, val = self.olhar()
            if (tipo, val) == ('op', '*'):
                self.pegar()
                f = self._mult(f, self.fator())
            elif (tipo, val) == ('op', '/'):
                self.pegar()
                g = self.fator()
                if g[0]:
                    self.erro('divisão por incógnita: o sistema precisa ser linear.')
                if g[1] == 0:
                    self.erro('divisão por zero.')
                f = ({v: a / g[1] for v, a in f[0].items()}, f[1] / g[1])
            elif tipo in ('num', 'var') or (tipo, val) == ('op', '('):
                f = self._mult(f, self.fator())  # multiplicação implícita: 2x, 2(x+y)
            else:
                return f

    def _mult(self, f, g):
        if f[0] and g[0]:
            self.erro('produto de incógnitas: o sistema precisa ser linear.')
        if not f[0]:
            f, g = g, f
        k = g[1]
        return ({v: a * k for v, a in f[0].items()}, f[1] * k)

    def fator(self):
        tipo, val = self.pegar()
        if tipo == 'num':
            return ({}, val)
        if tipo == 'var':
            return ({val: Fraction(1)}, Fraction(0))
        if (tipo, val) == ('op', '-'):
            c, k = self.fator()
            return ({v: -a for v, a in c.items()}, -k)
        if (tipo, val) == ('op', '+'):
            return self.fator()
        if (tipo, val) == ('op', '('):
            f = self.expr()
            if self.pegar() != ('op', ')'):
                self.erro('falta fechar o parêntese ")".')
            return f
        if tipo is None:
            self.erro('a expressão terminou antes do esperado.')
        self.erro(f'símbolo inesperado "{val}".')


def _ordenar_incognitas(nomes):
    nomes = list(dict.fromkeys(nomes))
    if all(v in 'xyzw' for v in nomes):
        return sorted(nomes, key='xyzw'.index)

    def chave(v):
        m = re.fullmatch(r'([A-Za-z]+)_?(\d+)', v)
        return (m.group(1), int(m.group(2))) if m else (v, -1)
    return sorted(nomes, key=chave)


def nomes_padrao(n):
    return list('xyz')[:n] if n <= 3 else [f'x{i}' for i in range(1, n + 1)]


def _linha_matriz(linha, nlin):
    s = re.sub(r'[\[\]{}()]', ' ', linha)
    if s.count('|') > 1:
        raise ValueError(f'Linha {nlin}: use no máximo uma barra "|".')
    s = s.replace('|', ' | ')
    if ';' in s:
        pedacos = [p for p in re.split(r'[;\s]+', s) if p]
    else:
        pedacos = []
        partes = s.split()
        for p in partes:
            p = p.strip(',')
            if p.count(',') == 1 and re.fullmatch(r'[+-]?\d+,\d+(?:[eE][+-]?\d+)?', p) and len(partes) > 1:
                pedacos.append(p)          # vírgula decimal: 0,5
            else:
                pedacos.extend(q for q in p.split(',') if q)
    valores, barra = [], None
    for p in pedacos:
        if p == '|':
            barra = len(valores)
            continue
        try:
            valores.append(_fracao(p))
        except (ValueError, ZeroDivisionError):
            raise ValueError(f'Linha {nlin}: "{p}" não é um número.') from None
    return valores, barra


def ler_sistema(texto):
    """Converte o texto digitado em um Sistema (A, b, nomes das incógnitas)."""
    linhas = []
    for nlin, bruta in enumerate(texto.splitlines(), 1):
        linha = bruta.split('#')[0]
        for a, b in _TROCAS.items():
            linha = linha.replace(a, b)
        linha = linha.translate(_SUBSCRITOS)
        if linha.strip():
            linhas.append((nlin, linha))
    if not linhas:
        raise ValueError('Digite o sistema: uma equação por linha (ou uma linha da matriz aumentada).')
    com_igual = ['=' in l for _, l in linhas]
    if all(com_igual):
        eqs = []
        for nlin, linha in linhas:
            toks = _tokenizar(linha, nlin)
            iguais = [i for i, t in enumerate(toks) if t == ('op', '=')]
            if len(iguais) != 1:
                raise ValueError(f'Linha {nlin}: cada equação deve ter exatamente um sinal "=".')
            k = iguais[0]
            if k == 0 or k == len(toks) - 1:
                raise ValueError(f'Linha {nlin}: falta um dos lados da equação.')
            ce, ke = _Expressao(toks[:k], nlin).completa()
            cd, kd = _Expressao(toks[k + 1:], nlin).completa()
            coefs = dict(ce)
            for v, a in cd.items():
                coefs[v] = coefs.get(v, 0) - a
            eqs.append((coefs, kd - ke))
        nomes = _ordenar_incognitas([v for c, _ in eqs for v in c])
        if len(nomes) != len(eqs):
            raise ValueError(f'O sistema precisa ser quadrado: encontrei {len(eqs)} equação(ões) e '
                             f'{len(nomes)} incógnita(s) ({", ".join(nomes) or "nenhuma"}).')
        A = [[c.get(v, Fraction(0)) for v in nomes] for c, _ in eqs]
        b = [k for _, k in eqs]
    elif not any(com_igual):
        n = len(linhas)
        A, b = [], []
        for nlin, linha in linhas:
            if re.search(r'[A-DF-Za-df-z]', linha):
                raise ValueError(f'Linha {nlin}: equações precisam de "="; linhas da matriz só têm números.')
            valores, barra = _linha_matriz(linha, nlin)
            if len(valores) != n + 1:
                raise ValueError(f'Linha {nlin}: com {n} linhas, cada linha da matriz aumentada precisa de '
                                 f'{n + 1} números ({n} coeficientes + termo independente); encontrei {len(valores)}.')
            if barra is not None and barra != n:
                raise ValueError(f'Linha {nlin}: a barra "|" deve vir antes do último número (termo independente).')
            A.append(valores[:n])
            b.append(valores[n])
        nomes = nomes_padrao(n)
    else:
        raise ValueError('Use um único formato: ou todas as linhas são equações (com "="), '
                         'ou todas são linhas da matriz aumentada.')
    if len(b) > N_MAX:
        raise ValueError(f'Para a animação use no máximo {N_MAX} equações (n ≤ {N_MAX}).')
    return Sistema(A, b, nomes)


def _num_texto(v):
    v = Fraction(v)
    return str(v.numerator) if v.denominator == 1 else f'{v.numerator}/{v.denominator}'


def equacao_texto(coefs, rhs, nomes, forcar=()):
    termos = []
    for j, (a, v) in enumerate(zip(coefs, nomes)):
        a = Fraction(a)
        if a == 0 and j not in forcar:
            continue
        m = abs(a)
        c = '' if m == 1 else (_num_texto(m) if m.denominator == 1 else f'({_num_texto(m)})')
        termos.append(('-' if a < 0 else '+', c + v))
    if not termos:
        esq = '0'
    else:
        esq = ('-' if termos[0][0] == '-' else '') + termos[0][1]
        esq += ''.join(f' {s} {t}' for s, t in termos[1:])
    return f'{esq} = {_num_texto(rhs)}'


def sistema_para_texto(sis, formato='equacoes'):
    if formato == 'matriz':
        larg = max(len(_num_texto(v)) for linha in sis.A for v in linha)
        return '\n'.join('  '.join(_num_texto(v).rjust(larg) for v in linha) + '  |  ' + _num_texto(bi)
                         for linha, bi in zip(sis.A, sis.b))
    n = sis.n
    nulas = {j for j in range(n) if all(sis.A[i][j] == 0 for i in range(n))}
    return '\n'.join(equacao_texto(linha, bi, sis.nomes, forcar=nulas if i == 0 else ())
                     for i, (linha, bi) in enumerate(zip(sis.A, sis.b)))


def sistema_aleatorio(n, semente=None):
    """Sistema n×n com solução inteira e contas 'amigáveis' (A = L·U com inteiros pequenos)."""
    rng = random.Random(semente)
    while True:
        L = [[1 if i == j else (rng.randint(-2, 2) if j < i else 0) for j in range(n)] for i in range(n)]
        U = [[rng.choice([1, 2, 3, -1, -2]) if i == j else (rng.randint(-3, 3) if j > i else 0)
              for j in range(n)] for i in range(n)]
        A = [[sum(L[i][t] * U[t][j] for t in range(n)) for j in range(n)] for i in range(n)]
        x = [rng.randint(-5, 5) for _ in range(n)]
        if max(abs(v) for linha in A for v in linha) > 20 or not any(x):
            continue
        b = [sum(A[i][j] * x[j] for j in range(n)) for i in range(n)]
        return Sistema(A, b, nomes_padrao(n))


def sistema_hilbert(n):
    """Matriz de Hilbert (mal condicionada); b escolhido para que a solução exata seja (1, 1, ..., 1)."""
    A = [[Fraction(1, i + j + 1) for j in range(n)] for i in range(n)]
    return Sistema(A, [sum(linha) for linha in A], nomes_padrao(n))


EXEMPLOS = {
    'Exemplo do material (3×3)': '2x + y - z = 8\n-3x - y + 2z = -11\n-2x + y + 2z = -3',
    'Pivô nulo (2×2)': '0x + 2y = 4\nx + y = 3',
    'Matriz da fatoração LU (3×3)': '2x - y + z = 2\n4x + y - z = 8\n-2x + 2y + 3z = -3',
    'Sistema 4×4': 'x1 + 2x2 - x3 + x4 = 6\n2x1 + 3x2 - 2x3 + 2x4 = 10\n'
                   '-x1 + x2 + 3x3 - 2x4 = -2\n3x1 - x2 + x3 + 4x4 = 12',
    'Sistema 5×5 (matriz aumentada)': None,
    'Pivô muito pequeno (compare float com/sem pivoteamento)':
        '# matriz aumentada [A | b]; o pivô 2e-14 é quase zero\n2e-14  1 | 1\n1      1 | 2',
    'Infinitas soluções (SPI)': 'x + y + z = 6\n2x + 2y + 2z = 12\nx - y + z = 2',
    'Sem solução (SI)': 'x + y + z = 6\n2x + 2y + 2z = 10\nx - y + z = 2',
    'Aleatório n×n': lambda n, s: sistema_para_texto(sistema_aleatorio(n, s)),
    'Hilbert n×n (mal condicionada)': lambda n, s: sistema_para_texto(sistema_hilbert(n), 'matriz'),
}
EXEMPLOS['Sistema 5×5 (matriz aumentada)'] = sistema_para_texto(sistema_aleatorio(5, 12), 'matriz')


# --------------------------------------------------------------------------
# 2) Aritmética (frações exatas ou float64) e formatação dos números
# --------------------------------------------------------------------------
def _float_tex(x):
    if x != x:
        return r'\mathrm{NaN}'
    if x in (float('inf'), float('-inf')):
        return ('-' if x < 0 else '') + r'\infty'
    if x == 0:
        return '0'
    s = f'{x:.4g}'
    if 'e' in s:
        m, e = s.split('e')
        return (f'{m}\\times' if m not in ('1', '-1') else ('-' if m == '-1' else '')) + f'10^{{{int(e)}}}'
    return s


class Aritmetica:
    """Como os números são guardados: frações exatas (Fraction) ou ponto flutuante (float64)."""

    def __init__(self, exato=True, tol=1e-14):
        self.exato, self.tol = exato, tol

    def conv(self, v):
        return Fraction(v) if self.exato else float(v)

    def eh_zero(self, v):
        return v == 0 if self.exato else abs(v) <= self.tol

    def tex(self, v, par=False):
        """Número em notação matemática; par=True coloca negativos entre parênteses."""
        if v is None:
            return '?'
        if self.exato:
            v = Fraction(v)
            if v.denominator == 1:
                s = str(v.numerator)
            elif max(len(str(abs(v.numerator))), len(str(v.denominator))) > 6:
                s = r'\approx' + _float_tex(float(v))
            else:
                s = ('-' if v < 0 else '') + r'\frac{%d}{%d}' % (abs(v.numerator), v.denominator)
            neg = v < 0
        else:
            s = _float_tex(float(v))
            neg = s.startswith('-')
        return f'({s})' if par and neg else s

    def texto(self, v):
        if self.exato:
            return _num_texto(v)
        return f'{float(v):.6g}'


def nome_tex(v):
    m = re.fullmatch(r'([A-Za-z]+)_?(\d+)', v)
    return f'{m.group(1)}_{{{m.group(2)}}}' if m else v.replace('_', r'\_')


def _termo_tex(coef, nome, ar):
    if coef == 1:
        return nome
    if coef == -1:
        return '-' + nome
    return ar.tex(coef) + r'\,' + nome


def equacao_tex(coefs, rhs, nomes_tex, ar):
    partes = []
    for a, v in zip(coefs, nomes_tex):
        if a == 0:
            continue
        partes.append(('-' if a < 0 else '+', _termo_tex(abs(a), v, ar)))
    if not partes:
        esq = '0'
    else:
        esq = ('-' if partes[0][0] == '-' else '') + partes[0][1]
        esq += ''.join(f' {s} {t}' for s, t in partes[1:])
    return f'{esq} = {ar.tex(rhs)}'


def resolver_exato(sis):
    """Solução exata com frações (None se A for singular) — usada para medir o erro do float."""
    n = sis.n
    M = [linha[:] + [bi] for linha, bi in zip(sis.A, sis.b)]
    for k in range(n):
        p = next((i for i in range(k, n) if M[i][k] != 0), None)
        if p is None:
            return None
        M[k], M[p] = M[p], M[k]
        for i in range(k + 1, n):
            f = M[i][k] / M[k][k]
            for j in range(k, n + 1):
                M[i][j] -= f * M[k][j]
    x = [Fraction(0)] * n
    for i in reversed(range(n)):
        x[i] = (M[i][n] - sum(M[i][j] * x[j] for j in range(i + 1, n))) / M[i][i]
    return x


# --------------------------------------------------------------------------
# 3) Registro dos passos: cada operação vira um "quadro" da animação
# --------------------------------------------------------------------------
NIVEIS = {'detalhado': 'Detalhado (3 quadros por operação)',
          'operacao': 'Uma operação por quadro',
          'coluna': 'Uma coluna por quadro'}
TODOS = frozenset(NIVEIS)
DET = frozenset({'detalhado'})
DET_OP = frozenset({'detalhado', 'operacao'})
OP = frozenset({'operacao'})
COL = frozenset({'coluna'})


def _grupos(itens, k):
    return [itens[i:i + k] for i in range(0, len(itens), k)]


def gerar_passos(sistema, pivoteamento=False, exato=True, tol=1e-14):
    """Executa a eliminação de Gauss + substituição regressiva registrando cada operação.

    Retorna (quadros, resultado). Cada quadro guarda uma cópia da matriz aumentada
    naquele instante, o texto explicativo e o que deve ser destacado no desenho.
    """
    ar = Aritmetica(exato, tol)
    T = ar.tex
    n = sistema.n
    nomes = [nome_tex(v) for v in sistema.nomes]
    M = [[ar.conv(v) for v in linha] + [ar.conv(bi)] for linha, bi in zip(sistema.A, sistema.b)]
    x = [None] * n
    eliminados = set()
    quadros = []
    resultado = dict(status=None, trocas=[], multiplicadores={}, pivos=[], x=None, residuo=None)

    def P(v):
        return ar.tex(v, par=True)

    def a(i, j):
        return f'a_{{{i + 1}{j + 1}}}'

    def R(i):
        return f'R_{{{i + 1}}}'

    def quadro(tipo, titulo, texto, niveis=TODOS, **extra):
        q = dict(tipo=tipo, titulo=titulo, texto=list(texto), niveis=niveis,
                 M=[linha[:] for linha in M], eliminados=frozenset(eliminados), x=list(x),
                 cel={}, lin={})
        q.update(extra)
        quadros.append(q)

    def bloco_sistema():
        return {'bloco': [f'${equacao_tex(M[i][:n], M[i][n], nomes, ar)}$' for i in range(n)],
                'colunas': 1 if n <= 4 else 2}

    quadro('inicio', 'Sistema original e sua matriz aumentada [A | b]',
           [bloco_sistema(),
            'Cada linha da matriz é uma equação; a linha tracejada separa os coeficientes dos termos independentes.'])

    r = 0  # linha onde está o pivô da etapa atual
    for k in range(n):
        if r >= n:
            break
        etapa = k + 1
        candidatos = list(range(r, n))

        def sem_pivo():
            for i in candidatos:
                eliminados.add((i, k))
            quadro('sem_pivo', f'Etapa {etapa}: a coluna {k + 1} não tem pivô',
                   [f'Todos os elementos da coluna {k + 1}, da linha {r + 1} para baixo, são zero'
                    + ('' if exato else r' (em módulo, $\leq 10^{-14}$)') + '.',
                    'Não há pivô nesta coluna: a matriz A é singular e o sistema não tem solução única.',
                    'Seguimos para a próxima coluna, mantendo a mesma linha de pivô.'],
                   cel={(i, k): 'erro' for i in candidatos})

        if len(candidatos) == 1:  # última linha: nada abaixo para eliminar
            if ar.eh_zero(M[r][k]):
                sem_pivo()
                continue
            resultado['pivos'].append((r, k))
            r += 1
            continue

        trocou = None
        if pivoteamento:
            p = max(candidatos, key=lambda i: abs(M[i][k]))  # igual a np.argmax: 1º maior
            if ar.eh_zero(M[p][k]):
                sem_pivo()
                continue
            mods = [f'$|{T(M[i][k])}| = {T(abs(M[i][k]))}$' for i in candidatos]
            texto = [f'Pivoteamento parcial: na coluna {k + 1}, da linha {r + 1} até a linha {n}, comparamos os módulos:']
            texto += ['      '.join(g) for g in _grupos(mods, 4)]
            if p != r:
                texto.append(f'O maior módulo está na linha {p + 1}: trocamos ${R(r)} \\leftrightarrow {R(p)}$ '
                             f'para que ${T(M[p][k])}$ seja o pivô.')
            else:
                texto.append(f'O maior módulo já está na linha {r + 1}: não é preciso trocar linhas. '
                             f'Pivô: ${a(r, k)} = {T(M[r][k])}$.')
            cel = {(i, k): 'candidato' for i in candidatos}
            cel[(p, k)] = 'maximo'
            quadro('escolha', f'Etapa {etapa}: escolha do pivô da coluna {k + 1}', texto, DET_OP, cel=cel)
            if p != r:
                d = p - r
                for t in (0.2, 0.4, 0.6, 0.8):
                    quadro('troca_anim', f'Trocando as linhas ${R(r)} \\leftrightarrow {R(p)}$',
                           ['A troca só muda a ordem das equações: o conjunto solução não se altera.'],
                           DET, lin={r: 'troca', p: 'troca'}, desloc={r: t * d, p: -t * d}, seta=('troca', r, p))
                M[r], M[p] = M[p], M[r]
                trocou = (r, p)
                resultado['trocas'].append((r, p))
                quadro('troca', f'Troca de linhas: ${R(r)} \\leftrightarrow {R(p)}$',
                       [f'As linhas {r + 1} e {p + 1} trocaram de lugar. A troca só muda a ordem das equações: '
                        'o conjunto solução não se altera.',
                        f'Novo pivô: ${a(r, k)} = {T(M[r][k])}$.'],
                       DET_OP, lin={r: 'troca', p: 'troca'}, cel={(r, k): 'pivo'}, seta=('troca', r, p))
        else:
            if ar.eh_zero(M[r][k]):
                abaixo_nz = [i for i in candidatos[1:] if not ar.eh_zero(M[i][k])]
                if abaixo_nz:
                    i0 = abaixo_nz[0]
                    cel = {(r, k): 'erro'}
                    cel.update({(i, k): 'candidato' for i in abaixo_nz})
                    quadro('pivo_nulo', 'Pivô nulo: a eliminação sem pivoteamento para aqui',
                           [f'O pivô seria ${a(r, k)} = {T(M[r][k])}$'
                            + ('' if exato or M[r][k] == 0 else r' (em módulo $\leq 10^{-14}$: tratado como zero)') + '.',
                            f'Para zerar ${a(i0, k)}$ precisaríamos de $m_{{{i0 + 1}{k + 1}}} = {a(i0, k)} \\div {a(r, k)}$: '
                            'divisão por zero!',
                            f'Solução: trocar ${R(r)}$ com uma linha de baixo que tenha elemento não nulo na coluna {k + 1} '
                            f'(por exemplo, ${R(i0)}$).',
                            'Marque "Pivoteamento parcial" para o algoritmo fazer essa troca automaticamente.'],
                           cel=cel, lin={r: 'erro'})
                    resultado['status'] = 'pivo_nulo'
                    return quadros, resultado
                sem_pivo()
                continue

        piv = M[r][k]
        abaixo = candidatos[1:]
        if not pivoteamento:
            cel = {(r, k): 'pivo'}
            cel.update({(i, k): 'alvo' for i in abaixo})
            quadro('pivo', f'Etapa {etapa}: eliminar os elementos abaixo do pivô',
                   [f'O pivô é ${a(r, k)} = {T(piv)}$ (linha {r + 1}, coluna {k + 1}).',
                    'Vamos zerar os elementos abaixo dele na mesma coluna: '
                    + ', '.join(f'${T(M[i][k])}$' for i in abaixo) + '.',
                    f'Para cada linha $i$: $m_{{i{k + 1}}} = a_{{i{k + 1}}} \\div {a(r, k)}$ e '
                    f'$R_i \\leftarrow R_i - m_{{i{k + 1}}}\\,{R(r)}$.'],
                   DET_OP, cel=cel, lin={r: 'pivo'})

        resumo = []
        for i in abaixo:
            mt = f'm_{{{i + 1}{k + 1}}}'
            op = f'{R(i)} \\leftarrow {R(i)} - {mt}\\,{R(r)}'
            if M[i][k] == 0:
                eliminados.add((i, k))
                quadro('ja_zero', f'Linha {i + 1}: nada a fazer',
                       [f'O elemento ${a(i, k)}$ já é zero, então ${mt} = 0$ e a linha {i + 1} não muda.'],
                       DET_OP, cel={(r, k): 'pivo', (i, k): 'zerado'}, lin={r: 'pivo', i: 'alterada'})
                resumo.append(f'${mt} = 0$')
                continue
            antigo = M[i][k]
            m = antigo / piv
            resultado['multiplicadores'][(i + 1, k + 1)] = m
            txt_m = f'${mt} = \\frac{{{a(i, k)}}}{{{a(r, k)}}} = {P(antigo)} \\div {P(piv)} = {T(m)}$'
            aviso = []
            if abs(m) > 1e3:
                aviso = ['Atenção: pivô muito pequeno ⇒ multiplicador enorme. '
                         + ('Em ponto flutuante isso amplifica os erros de arredondamento.' if not exato
                            else 'Com frações exatas não há arredondamento; em ponto flutuante, haveria.')]
            quadro('multiplicador', f'Multiplicador ${mt}$',
                   [txt_m, f'Ele diz quantas vezes a linha do pivô (${R(r)}$) deve ser subtraída de ${R(i)}$ '
                           f'para zerar ${a(i, k)}$.'] + aviso,
                   DET, cel={(r, k): 'pivo', (i, k): 'alvo'}, lin={r: 'pivo', i: 'alterada'})
            aux = (f'{mt}\\,{R(r)}', [m * v for v in M[r]])
            txt_p = [f'Primeiro calculamos a linha auxiliar ${mt}\\,{R(r)} = {P(m)}\\cdot {R(r)}$ (abaixo da matriz).',
                     f'Depois subtraímos essa linha de ${R(i)}$, elemento a elemento, inclusive o termo independente.']
            if m < 0:
                txt_p.append(f'Como ${mt} < 0$, subtrair ${mt}\\,{R(r)}$ é o mesmo que somar ${T(-m)}\\,{R(r)}$.')
            quadro('produto', f'Operação ${op}$', txt_p, DET,
                   cel={(r, k): 'pivo', (i, k): 'alvo'}, lin={r: 'pivo', i: 'alterada'}, aux=aux, seta=('op', r, i))
            for j in range(k, n + 1):  # mesma conta do material: U[i,k:] -= fator*U[k,k:]
                M[i][j] = M[i][j] - m * M[r][j]
            eliminados.add((i, k))
            txt_z = f'O elemento ${a(i, k)}$ virou zero: ${T(antigo)} - {P(m)}\\cdot {P(piv)} = {T(M[i][k])}$.'
            cel_res = {(r, k): 'pivo', (i, k): 'zerado'}
            cel_res.update({(i, j): 'novo' for j in range(k + 1, n + 1)})
            quadro('resultado', f'Resultado de ${op}$',
                   [f'A nova linha {i + 1} é ${R(i)} - {mt}\\,{R(r)}$ (valores alterados em azul).', txt_z], DET,
                   cel=cel_res, lin={r: 'pivo', i: 'alterada'}, aux=aux, seta=('op', r, i))
            quadro('operacao', f'${op}$',
                   [txt_m, f'Subtraímos a linha auxiliar ${mt}\\,{R(r)}$ de ${R(i)}$, inclusive o termo independente.',
                    txt_z] + aviso, OP,
                   cel=cel_res, lin={r: 'pivo', i: 'alterada'}, aux=aux, seta=('op', r, i))
            resumo.append(f'${mt} = {T(m)}$')

        resultado['pivos'].append((r, k))
        txt = []
        if pivoteamento:
            txt.append('Pivoteamento parcial: ' + (
                f'trocamos ${R(trocou[0])} \\leftrightarrow {R(trocou[1])}$ para usar o maior pivô em módulo.'
                if trocou else 'o maior módulo já estava na posição do pivô (sem troca).'))
        txt.append(f'Pivô ${a(r, k)} = {T(piv)}$. Multiplicadores usados em '
                   f'$R_i \\leftarrow R_i - m_{{i{k + 1}}}\\,{R(r)}$:')
        txt += ['      '.join(g) for g in _grupos(resumo, 4)]
        txt.append(f'A coluna {k + 1} ficou zerada abaixo do pivô.')
        cel = {(r, k): 'pivo'}
        cel.update({(i, k): 'zerado' for i in abaixo})
        lin = {r: 'pivo'}
        lin.update({i: 'alterada' for i in abaixo})
        quadro('resumo_coluna', f'Etapa {etapa}: eliminação na coluna {k + 1}', txt, COL, cel=cel, lin=lin)
        r += 1

    posto = len(resultado['pivos'])
    diag = {(i, j): 'pivo' for i, j in resultado['pivos']}
    if posto < n:  # matriz singular: classifica o sistema
        nulas = list(range(posto, n))
        contra = [i for i in nulas if not ar.eh_zero(M[i][n])]
        cel = dict(diag)
        if contra:
            i = contra[0]
            cel.update({(i, j): 'erro' for j in range(n + 1)})
            resultado['status'] = 'impossivel'
            quadro('impossivel', 'Sistema impossível (SI): não existe solução',
                   [f'A eliminação terminou com linhas de coeficientes nulos (posto de A = {posto} < n = {n}).',
                    f'A linha {i + 1} representa a equação $0 = {T(M[i][n])}$, que é impossível.',
                    'Conclusão: o sistema não tem solução.'], cel=cel, lin={i: 'erro'})
        else:
            cel.update({(i, j): 'candidato' for i in nulas for j in range(n + 1)})
            resultado['status'] = 'indeterminado'
            quadro('indeterminado', 'Sistema possível e indeterminado (SPI): infinitas soluções',
                   [f'A eliminação terminou com {len(nulas)} linha(s) do tipo $0 = 0$ (posto de A = {posto} < n = {n}).',
                    'Essas equações não trazem informação nova: sobram incógnitas livres.',
                    'Conclusão: infinitas soluções; não há solução única para a substituição regressiva.'], cel=cel)
        return quadros, resultado

    quadro('triangular', 'Matriz triangular superior: fim da eliminação',
           ['Todos os elementos abaixo da diagonal principal são zero. O sistema equivalente é:',
            bloco_sistema(),
            'Agora resolvemos de baixo para cima (substituição regressiva): '
            r'$x_i = (c_i - \sum_{j>i} u_{ij}\,x_j) \div u_{ii}$'], cel=diag)

    for i in reversed(range(n)):
        u, c = M[i][i], M[i][n]
        conhecidos = [j for j in range(i + 1, n) if M[i][j] != 0]
        soma = ar.conv(0)
        for j in range(i + 1, n):
            soma = soma + M[i][j] * x[j]
        x[i] = (c - soma) / u
        termo = _termo_tex(u, nomes[i], ar)
        eq = termo
        for j in conhecidos:
            coef = abs(M[i][j])
            eq += (' - ' if M[i][j] < 0 else ' + ') + ('' if coef == 1 else T(coef) + r'\cdot ') + P(x[j])
        linhas = [f'Linha {i + 1}: ${eq} = {T(c)}$']
        if conhecidos:
            linhas.append(f'${termo} = {T(c)} - {P(soma)} = {T(c - soma)}$')
        linhas.append(f'${nomes[i]} = {P(c - soma)} \\div {P(u)} = {T(x[i])}$' if u != 1
                      else f'$\\Rightarrow\\; {nomes[i]} = {T(x[i])}$')
        cel = {(i, i): 'calculando', (i, n): 'novo'}
        cel.update({(i, j): 'conhecido' for j in conhecidos})
        quadro('regressiva', f'Substituição regressiva: calculando ${nomes[i]}$', linhas,
               cel=cel, lin={i: 'regressiva'}, sol_destaque=i)

    resultado['status'] = 'solucao'
    resultado['x'] = list(x)
    A0 = [[ar.conv(v) for v in linha] for linha in sistema.A]
    b0 = [ar.conv(v) for v in sistema.b]
    res = []
    for i in range(n):
        s = ar.conv(0)
        for j in range(n):
            s = s + A0[i][j] * x[j]
        res.append(b0[i] - s)
    resultado['residuo'] = res
    texto = ['Solução encontrada:',
             {'bloco': [f'${nomes[i]} = {T(x[i])}$' for i in range(n)], 'colunas': min(n, 4)}]
    if exato:
        texto.append(r'Conferência nas equações originais: $b - A\mathbf{x} = ('
                     + r',\,'.join(T(v) for v in res) + ')$ — todas satisfeitas.')
    else:
        nr = float(np.linalg.norm(np.array(res, dtype=float)))
        texto.append(f'Resíduo: $\\Vert b - A\\hat{{x}} \\Vert_2 = {_float_tex(nr)}$ '
                     r'(mede o quanto $\hat{x}$ satisfaz as equações).')
        xe = resolver_exato(sistema)
        if xe is not None:
            err = float(np.linalg.norm(np.array(x, dtype=float) - np.array(xe, dtype=float)))
            resultado['erro'] = err
            texto.append(f'Erro em relação à solução exata (calculada com frações): '
                         f'$\\Vert \\hat{{x}} - x \\Vert_2 = {_float_tex(err)}$.')
    try:
        An, bn = sistema.como_numpy()
        xn = np.linalg.solve(An, bn)
        texto.append('Conferência com NumPy (np.linalg.solve): '
                     + np.array2string(xn, precision=4, suppress_small=True, separator=', ', max_line_width=200))
    except np.linalg.LinAlgError:
        pass
    quadro('solucao', 'Solução do sistema', texto, cel=diag)
    return quadros, resultado


# --------------------------------------------------------------------------
# 4) Desenho de um quadro com matplotlib
# --------------------------------------------------------------------------
U_POL = 0.5  # tamanho de uma "unidade" do desenho, em polegadas
COR_TEXTO, COR_SUAVE, COR_AZUL, COR_ROXO = '#212121', '#78909C', '#1565C0', '#6A1B9A'
COR_ELIMINADO = '#F1F8E9'
ESTILO_LINHA = {'pivo': '#FFF8E1', 'alterada': '#E3F2FD', 'troca': '#EDE7F6', 'regressiva': '#E3F2FD', 'erro': '#FFEBEE'}
ESTILO_CEL = {'pivo': '#FFD54F', 'alvo': '#EF9A9A', 'zerado': '#A5D6A7', 'novo': '#90CAF9',
              'candidato': '#FFE0B2', 'maximo': '#FFB74D', 'erro': '#FF8A80',
              'conhecido': '#C8E6C9', 'calculando': '#FFD54F'}
LEGENDA = [('pivo', 'pivô'), ('alvo', 'elemento a zerar'), ('zerado', 'zerado'), ('novo', 'valor alterado'),
           ('candidato', 'candidato a pivô'), ('conhecido', 'já calculado')]


def _n_linhas(texto):
    return sum(-(-len(it['bloco']) // it['colunas']) if isinstance(it, dict) else 1 for it in texto)


class Geometria:
    """Posições (em 'unidades' de 0,5 polegada) de cada parte do desenho para um sistema n×n."""

    def __init__(self, n, linhas_texto):
        self.n = n
        self.cw = 1.9 if n <= 4 else (1.75 if n <= 6 else 1.6)
        self.ch = 0.95 if n <= 5 else 0.85
        self.fs = 13 if n <= 4 else (12 if n <= 6 else 11)
        self.x_mat = 2.8
        self.larg = (n + 1) * self.cw
        self.x_sol = self.x_mat + self.larg + 1.3
        self.W = max(self.x_sol + 1.2 + self.cw + 0.5, 18.0)
        self.dl = 0.8
        self.H = 2.45 + n * self.ch + 0.35 + self.ch + 0.55 + linhas_texto * self.dl + 0.85
        topo = self.H
        self.y_tit, self.y_sub, self.y_barra = topo - 0.55, topo - 1.1, topo - 1.45
        self.y_cab = topo - 1.95
        self.y_mat = topo - 2.35
        self.y_aux = self.y_mat - n * self.ch - 0.35
        self.y_txt = self.y_aux - self.ch - 0.5
        self.figsize = (self.W * U_POL, self.H * U_POL)

    def xc(self, j):
        return self.x_mat + (j + 0.5) * self.cw

    def yc(self, i):
        return self.y_mat - (i + 0.5) * self.ch


def _caixa(ax, x, y, w, h, cor, z=1, borda=None, lw=0.0):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0,rounding_size=0.12',
                                facecolor=cor, edgecolor=borda or cor, linewidth=lw, zorder=z))


def desenhar_quadro(fig, anim, idx):
    q, g, ar = anim.quadros[idx], anim.geo, anim.ar
    n, nomes, total = g.n, anim.nomes_tex, len(anim.quadros)
    fig.clear()
    fig.patch.set_facecolor('white')
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, g.W)
    ax.set_ylim(0, g.H)
    ax.axis('off')
    ajustar = []  # (texto, largura máxima): textos longos são reduzidos para caber

    # título, subtítulo e barra de progresso
    t = ax.text(0.5, g.y_tit, q['titulo'], fontsize=15, fontweight='bold', va='center', color=COR_TEXTO)
    ajustar.append((t, g.W - 4.2))
    ax.text(g.W - 0.5, g.y_tit, f'passo {idx + 1} de {total}', fontsize=11, ha='right', va='center', color=COR_SUAVE)
    ax.text(0.5, g.y_sub, anim.subtitulo, fontsize=10.5, va='center', color=COR_SUAVE)
    ax.add_patch(Rectangle((0.5, g.y_barra), g.W - 1, 0.11, color='#ECEFF1', lw=0))
    ax.add_patch(Rectangle((0.5, g.y_barra), (g.W - 1) * (idx + 1) / total, 0.11, color='#1E88E5', lw=0))

    # cabeçalhos das colunas e rótulos das linhas
    for j in range(n + 1):
        ax.text(g.xc(j), g.y_cab, f'${nomes[j]}$' if j < n else '$b$', ha='center', va='center',
                fontsize=12, color=COR_SUAVE)
    for i in range(n):
        ax.text(g.x_mat - 0.3, g.yc(i), f'$R_{{{i + 1}}}$', ha='right', va='center', fontsize=12.5,
                color=COR_AZUL if q['lin'].get(i) else COR_SUAVE)

    # matriz aumentada (linhas em movimento são desenhadas por cima)
    desloc = q.get('desloc', {})
    for i in sorted(range(n), key=lambda i: i in desloc):
        z = 6 if i in desloc else 1
        ytop = g.y_mat - (i + desloc.get(i, 0)) * g.ch
        est_l = q['lin'].get(i)
        if est_l or i in desloc:
            _caixa(ax, g.x_mat - 0.06, ytop - g.ch + 0.03, g.larg + 0.12, g.ch - 0.06,
                   ESTILO_LINHA.get(est_l, 'white'), z)
        for j in range(n + 1):
            x0 = g.x_mat + j * g.cw
            est = q['cel'].get((i, j))
            antigo = (i, j) in q['eliminados'] and est is None
            cor = ESTILO_CEL.get(est) or (COR_ELIMINADO if antigo else None)
            if cor:
                _caixa(ax, x0 + 0.09, ytop - g.ch + 0.09, g.cw - 0.18, g.ch - 0.18, cor, z + 1,
                       borda='#E65100' if est == 'maximo' else None, lw=2.2 if est == 'maximo' else 0)
            t = ax.text(x0 + g.cw / 2, ytop - g.ch / 2, f'${ar.tex(q["M"][i][j])}$', ha='center', va='center',
                        fontsize=g.fs, color='#9E9E9E' if antigo else COR_TEXTO, zorder=z + 2)
            ajustar.append((t, g.cw - 0.25))

    # colchetes e separador [A | b]
    y0, y1 = g.y_mat - n * g.ch, g.y_mat
    for xb, d in ((g.x_mat - 0.12, 1), (g.x_mat + g.larg + 0.12, -1)):
        ax.plot([xb, xb], [y0, y1], color='#37474F', lw=1.8, zorder=8)
        ax.plot([xb, xb + d * 0.25], [y1, y1], color='#37474F', lw=1.8, zorder=8)
        ax.plot([xb, xb + d * 0.25], [y0, y0], color='#37474F', lw=1.8, zorder=8)
    xs = g.x_mat + n * g.cw
    ax.plot([xs, xs], [y0 + 0.08, y1 - 0.08], color='#37474F', lw=1.2, ls=(0, (4, 3)), zorder=8)

    # linha auxiliar m·R_k
    if q.get('aux'):
        rot, vals = q['aux']
        yb = g.y_aux - g.ch
        _caixa(ax, g.x_mat - 0.06, yb + 0.03, g.larg + 0.12, g.ch - 0.06, '#F3E5F5', 1)
        t = ax.text(g.x_mat - 0.3, yb + g.ch / 2, f'${rot} =$', ha='right', va='center', fontsize=12, color=COR_ROXO)
        ajustar.append((t, g.x_mat - 0.4))
        for j, v in enumerate(vals):
            t = ax.text(g.xc(j), yb + g.ch / 2, f'${ar.tex(v)}$', ha='center', va='center', fontsize=g.fs,
                        color=COR_ROXO)
            ajustar.append((t, g.cw - 0.25))

    # setas: operação R_i <- R_i - m R_k  ou troca de linhas
    seta = q.get('seta')
    if seta:
        tipo, r1, r2 = seta
        ya, yb2 = g.yc(r1), g.yc(r2)
        rad = min(0.5, 1.6 / max(abs(ya - yb2), 1e-9))
        ax.add_patch(FancyArrowPatch((g.x_mat - 1.15, ya), (g.x_mat - 1.15, yb2),
                                     connectionstyle=f'arc3,rad={rad}',
                                     arrowstyle='<|-|>' if tipo == 'troca' else '-|>', mutation_scale=16,
                                     color=COR_ROXO if tipo == 'troca' else COR_AZUL, lw=1.8, zorder=9))

    # painel da solução
    xs0 = g.x_sol + 1.0
    ax.text(xs0 + g.cw / 2, g.y_cab, 'solução', ha='center', va='center', fontsize=11, color=COR_SUAVE)
    for i in range(n):
        v = q['x'][i]
        est = 'calculando' if q.get('sol_destaque') == i else ('conhecido' if v is not None else None)
        _caixa(ax, xs0 + 0.05, g.y_mat - (i + 1) * g.ch + 0.09, g.cw - 0.1, g.ch - 0.18,
               ESTILO_CEL[est] if est else '#FAFAFA', 1, borda='#CFD8DC', lw=0.8)
        ax.text(xs0 - 0.12, g.yc(i), f'${nomes[i]} =$', ha='right', va='center', fontsize=12, color=COR_TEXTO)
        t = ax.text(xs0 + g.cw / 2, g.yc(i), f'${ar.tex(v)}$' if v is not None else '?', ha='center', va='center',
                    fontsize=g.fs, color=COR_TEXTO if v is not None else '#B0BEC5')
        ajustar.append((t, g.cw - 0.25))

    # texto explicativo
    y = g.y_txt
    for item in q['texto']:
        if isinstance(item, dict):
            k = item['colunas']
            largura = (g.W - 1.1) / k
            for grupo in _grupos(item['bloco'], k):
                for c, s in enumerate(grupo):
                    t = ax.text(0.9 + c * largura, y, s, va='top', fontsize=12.5, color=COR_TEXTO)
                    ajustar.append((t, largura - 0.5))
                y -= g.dl
        else:
            t = ax.text(0.55, y, item, va='top', fontsize=12.5, color=COR_TEXTO)
            ajustar.append((t, g.W - 1.1))
            y -= g.dl

    # legenda das cores
    passo_leg = (g.W - 1.0) / len(LEGENDA)
    for c, (est, nome) in enumerate(LEGENDA):
        xl = 0.5 + c * passo_leg
        _caixa(ax, xl, 0.28, 0.42, 0.32, ESTILO_CEL[est], 1)
        t = ax.text(xl + 0.55, 0.44, nome, va='center', fontsize=9.5, color=COR_SUAVE)
        ajustar.append((t, passo_leg - 0.7))

    # reduz a fonte dos textos que não couberam
    rend = fig.canvas.get_renderer()
    px = fig.dpi * U_POL
    for t, max_u in ajustar:
        w = t.get_window_extent(rend).width
        if w > max_u * px:
            t.set_fontsize(max(6.0, t.get_fontsize() * max_u * px / w))


class AnimacaoGauss:
    """Guarda os quadros de uma eliminação e sabe desenhá-los (PNG, GIF ou HTML animado)."""

    def __init__(self, sistema, pivoteamento=False, exato=True, nivel='detalhado', dpi=100):
        if isinstance(sistema, str):
            sistema = ler_sistema(sistema)
        self.sistema, self.pivoteamento, self.exato, self.nivel, self.dpi = sistema, pivoteamento, exato, nivel, dpi
        todos, self.resultado = gerar_passos(sistema, pivoteamento, exato)
        self.quadros = [q for q in todos if nivel in q['niveis']]
        self.ar = Aritmetica(exato)
        self.nomes_tex = [nome_tex(v) for v in sistema.nomes]
        self.geo = Geometria(sistema.n, max(_n_linhas(q['texto']) for q in self.quadros))
        n = sistema.n
        self.subtitulo = (f'Sistema {n}×{n}   ·   {"com" if pivoteamento else "sem"} pivoteamento parcial   ·   '
                          f'{"frações exatas" if exato else "ponto flutuante (float64)"}   ·   {NIVEIS[nivel].lower()}')
        self._cache = {}

    def __len__(self):
        return len(self.quadros)

    def figura(self, idx):
        fig = Figure(figsize=self.geo.figsize, dpi=self.dpi)
        FigureCanvasAgg(fig)
        desenhar_quadro(fig, self, idx)
        return fig

    def png(self, idx):
        if idx not in self._cache:
            buf = io.BytesIO()
            self.figura(idx).savefig(buf, format='png', dpi=self.dpi, facecolor='white')
            self._cache[idx] = buf.getvalue()
        return self._cache[idx]

    def salvar_gif(self, caminho='eliminacao_gauss.gif', ms=1500):
        from PIL import Image
        imgs = [Image.open(io.BytesIO(self.png(i))).convert('RGB') for i in range(len(self))]
        imgs[0].save(caminho, save_all=True, append_images=imgs[1:], duration=ms, loop=0)
        return caminho

    def html(self, ms=1500):
        fig = Figure(figsize=self.geo.figsize, dpi=80)
        FigureCanvasAgg(fig)
        anim = animation.FuncAnimation(fig, lambda i: desenhar_quadro(fig, self, i), frames=len(self),
                                       interval=ms, repeat=False)
        return anim.to_jshtml(default_mode='once')

## 2. Painel interativo

In [3]:
# ==========================================================================
# Painel interativo: edite o sistema e a animação é recalculada na hora
# ==========================================================================
class PainelGauss:
    def __init__(self, exemplo='Exemplo do material (3×3)', pivoteamento=False, exato=True, nivel='detalhado'):
        self.anim, self.sistema, self._chave = None, None, None
        self._travado = False      # evita que um widget dispare o outro em cascata
        self._suprimir = False     # evita redesenhos enquanto ajustamos o controle de passo
        self._passo_zero = True
        self._semente = 1
        est, L = {'description_width': 'initial'}, widgets.Layout
        self.w_exemplo = widgets.Dropdown(options=list(EXEMPLOS), value=exemplo, description='Exemplo:',
                                          style=est, layout=L(width='390px'))
        self.w_n = widgets.IntSlider(value=3, min=1, max=N_MAX, description='Dimensão n:', style=est,
                                     continuous_update=False, layout=L(width='270px'))
        self.w_sortear = widgets.Button(description='Sortear outro', icon='random', layout=L(width='140px'))
        self.w_texto = widgets.Textarea(rows=6, continuous_update=True, layout=L(width='98%'),
                                        placeholder='Ex.: 2x + y - z = 8   (uma equação por linha)   ou   2 1 -1 | 8')
        self.w_piv = widgets.Checkbox(value=pivoteamento, description='Pivoteamento parcial', indent=False,
                                      layout=L(width='190px'))
        self.w_arit = widgets.ToggleButtons(options=[('Frações exatas', True), ('Ponto flutuante', False)],
                                            value=exato, style={'button_width': '130px'})
        self.w_nivel = widgets.Dropdown(options=[(v, k) for k, v in NIVEIS.items()], value=nivel,
                                        description='Detalhe:', style=est, layout=L(width='340px'))
        self.w_play = widgets.Play(value=0, min=0, max=1, step=1, interval=1500)
        self.w_passo = widgets.IntSlider(value=0, min=0, max=1, description='Passo:', style=est,
                                         layout=L(width='380px'))
        widgets.link((self.w_play, 'value'), (self.w_passo, 'value'))
        self.w_ant = widgets.Button(icon='step-backward', tooltip='Passo anterior', layout=L(width='44px'))
        self.w_prox = widgets.Button(icon='step-forward', tooltip='Próximo passo', layout=L(width='44px'))
        self.w_vel = widgets.Dropdown(options=[('0,5×', 3000), ('1×', 1500), ('2×', 750), ('4×', 375)],
                                      value=1500, description='Velocidade:', style=est, layout=L(width='170px'))
        self.w_gif = widgets.Button(description='Salvar GIF', icon='download', layout=L(width='120px'))
        self.w_msg = widgets.HTML()
        self.w_img = widgets.Image(format='png', layout=L(max_width='100%', height='auto'))

        self.w_exemplo.observe(self._ao_mudar_exemplo, 'value')
        self.w_n.observe(self._ao_mudar_n, 'value')
        self.w_sortear.on_click(self._ao_sortear)
        self.w_texto.observe(self._ao_mudar_texto, 'value')
        self.w_piv.observe(lambda ch: self._recalcular(manter_passo=True), 'value')
        self.w_arit.observe(lambda ch: self._recalcular(manter_passo=True), 'value')
        self.w_nivel.observe(lambda ch: self._recalcular(manter_passo=False), 'value')
        self.w_passo.observe(lambda ch: self._desenhar(), 'value')
        self.w_ant.on_click(lambda b: self._andar(-1))
        self.w_prox.on_click(lambda b: self._andar(+1))
        self.w_vel.observe(lambda ch: setattr(self.w_play, 'interval', ch['new']), 'value')
        self.w_gif.on_click(self._ao_salvar_gif)

        wrap = L(flex_flow='row wrap', align_items='center')
        ajuda = widgets.HTML('<b>Sistema</b> — edite à vontade: uma equação por linha (ex.: <code>2x + y - z = 8</code>) '
                             'ou uma linha da matriz aumentada (ex.: <code>2 1 -1 | 8</code>). '
                             'Aceita frações (<code>1/2</code>), decimais e qualquer nome de incógnita.')
        self.ui = widgets.VBox([
            widgets.HBox([self.w_exemplo, self.w_n, self.w_sortear], layout=wrap),
            ajuda, self.w_texto,
            widgets.HBox([self.w_piv, self.w_arit, self.w_nivel], layout=wrap),
            widgets.HBox([self.w_play, self.w_ant, self.w_prox, self.w_passo, self.w_vel, self.w_gif], layout=wrap),
            self.w_msg, self.w_img])
        self._ao_mudar_exemplo()

    # ----- eventos -----
    def _ao_mudar_exemplo(self, change=None):
        if self._travado:
            return
        modelo = EXEMPLOS[self.w_exemplo.value]
        self._definir_texto(modelo(self.w_n.value, self._semente) if callable(modelo) else modelo)

    def _ao_mudar_n(self, change):
        if self._travado:
            return
        modelo = EXEMPLOS[self.w_exemplo.value]
        if not callable(modelo):
            self._travado = True
            self.w_exemplo.value = 'Aleatório n×n'
            self._travado = False
            modelo = EXEMPLOS['Aleatório n×n']
        self._definir_texto(modelo(change['new'], self._semente))

    def _ao_sortear(self, botao=None):
        self._semente += 1
        if self.w_exemplo.value != 'Aleatório n×n':
            self._travado = True
            self.w_exemplo.value = 'Aleatório n×n'
            self._travado = False
        self._definir_texto(EXEMPLOS['Aleatório n×n'](self.w_n.value, self._semente))

    def _definir_texto(self, texto):
        self._passo_zero = True
        if self.w_texto.value == texto:
            self._ao_mudar_texto()
        else:
            self.w_texto.value = texto  # dispara _ao_mudar_texto

    def _ao_mudar_texto(self, change=None):
        try:
            sis = ler_sistema(self.w_texto.value)
        except (ValueError, ZeroDivisionError) as e:
            extra = ' (abaixo continua a animação do último sistema válido)' if self.anim else ''
            self._mensagem(f'⚠️ {e}{extra}', erro=True)
            return
        if sis.n != self.w_n.value:
            self._travado = True
            self.w_n.value = sis.n
            self._travado = False
        self.sistema = sis
        self._recalcular(manter_passo=not self._passo_zero)
        self._passo_zero = False

    def _andar(self, delta):
        if self.anim:
            self.w_passo.value = min(max(self.w_passo.value + delta, 0), len(self.anim) - 1)

    def _recalcular(self, manter_passo=True):
        if self.sistema is None:
            return
        chave = (self.sistema.chave(), self.w_piv.value, self.w_arit.value, self.w_nivel.value)
        if chave == self._chave:
            self._resumo()
            return
        self._chave = chave
        passo = self.w_passo.value if manter_passo else 0
        self.anim = AnimacaoGauss(self.sistema, self.w_piv.value, self.w_arit.value, self.w_nivel.value)
        total = len(self.anim)
        self._suprimir = True
        self.w_passo.value = 0
        self.w_play.value = 0
        self.w_passo.max = self.w_play.max = total - 1
        self.w_passo.value = min(passo, total - 1)
        self.w_play.value = self.w_passo.value
        self._suprimir = False
        self._resumo()
        self._desenhar()

    def _desenhar(self):
        if self._suprimir or self.anim is None:
            return
        self.w_img.value = self.anim.png(min(self.w_passo.value, len(self.anim) - 1))

    def _resumo(self):
        res, ar, s = self.anim.resultado, self.anim.ar, self.sistema
        n = s.n
        base = f'Sistema {n}×{n} lido com sucesso — {len(self.anim)} quadros. '
        st = res['status']
        if st == 'solucao':
            base += 'Solução: ' + ', '.join(f'{v} = {ar.texto(xv)}' for v, xv in zip(s.nomes, res['x']))
        elif st == 'pivo_nulo':
            base += 'Sem pivoteamento a eliminação para num pivô nulo — marque "Pivoteamento parcial".'
        elif st == 'impossivel':
            base += 'Sistema impossível (sem solução).'
        else:
            base += 'Sistema possível e indeterminado (infinitas soluções).'
        self._mensagem(base)

    def _mensagem(self, texto, erro=False):
        cor = '#C62828' if erro else '#37474F'
        self.w_msg.value = f'<div style="color:{cor}; padding:4px 0">{html.escape(texto)}</div>'

    def _ao_salvar_gif(self, botao=None):
        if self.anim is None:
            return
        self._mensagem(f'Gerando GIF com {len(self.anim)} quadros...')
        caminho = self.anim.salvar_gif('eliminacao_gauss.gif', ms=self.w_vel.value)
        try:
            from google.colab import files
            files.download(caminho)
            self._mensagem(f'GIF gerado ({len(self.anim)} quadros): o download foi iniciado.')
        except ImportError:
            self._mensagem(f'GIF salvo em {os.path.abspath(caminho)}')

## 3. Animação

Edite a caixa de texto (uma equação por linha, ex.: `2x + y - z = 8`, ou uma linha da matriz aumentada, ex.: `2 1 -1 | 8`), mude a **Dimensão n** ou escolha um **Exemplo**. Use ▶ para reproduzir ou arraste **Passo**.

Cores: amarelo = pivô · vermelho = elemento a zerar · verde = zerado · azul = valor alterado · laranja = candidato a pivô.

In [ ]:
painel = PainelGauss()
display(painel.ui)

## 4. Conferência com o cálculo à mão e com as funções do material

O exemplo do material deve dar $m_{21}=-3/2$, $m_{31}=-1$, $m_{32}=4$ e a matriz triangular
$egin{bmatrix}2&1&-1&|&8\0&1/2&1/2&|&1\0&0&-1&|&1\end{bmatrix}$, com solução $x=2,\ y=3,\ z=-1$.

In [4]:
sis = ler_sistema(EXEMPLOS['Exemplo do material (3×3)'])
quadros, res = gerar_passos(sis, pivoteamento=False, exato=True)
U_anim = next(q['M'] for q in quadros if q['tipo'] == 'triangular')
print('Multiplicadores:', {f'm{i}{k}': str(m) for (i, k), m in res['multiplicadores'].items()})
print('Matriz triangular [U | c] da animação:')
for linha in U_anim:
    print('   ', [str(v) for v in linha])
assert U_anim == [[2, 1, -1, 8], [0, Fraction(1, 2), Fraction(1, 2), 1], [0, 0, -1, 1]]
assert res['x'] == [2, 3, -1]
print('✓ Igual à matriz obtida à mão; solução x = 2, y = 3, z = -1')

# Mesmas contas das funções do material (Gauss sem e com pivoteamento), para n = 2..8
rng = np.random.default_rng(2026)
testes = 0
for n in range(2, N_MAX + 1):
    for _ in range(10):
        A = rng.integers(-9, 10, (n, n)); b = rng.integers(-9, 10, n)
        if abs(np.linalg.det(A)) < 0.5:
            continue
        s = Sistema(A.tolist(), b.tolist(), nomes_padrao(n))
        for piv in (False, True):
            q, r = gerar_passos(s, pivoteamento=piv, exato=False)
            if r['status'] != 'solucao':
                continue
            x_ref = np.linalg.solve(A.astype(float), b.astype(float))
            assert np.allclose(r['x'], x_ref)
            testes += 1
print(f'✓ {testes} sistemas aleatórios (n = 2 a {N_MAX}) conferidos com np.linalg.solve')

Multiplicadores: {'m21': '-3/2', 'm31': '-1', 'm32': '4'}
Matriz triangular [U | c] da animação:
    ['2', '1', '-1', '8']
    ['0', '1/2', '1/2', '1']
    ['0', '0', '-1', '1']
✓ Igual à matriz obtida à mão; solução x = 2, y = 3, z = -1
✓ 135 sistemas aleatórios (n = 2 a 8) conferidos com np.linalg.solve


## 5. Animação exportável (funciona mesmo sem o painel)

Gera uma animação HTML com botões de reprodução. Troque o texto, `pivoteamento`, `exato` ou `nivel` (`'detalhado'`, `'operacao'`, `'coluna'`) para outras versões.

In [5]:
anim = AnimacaoGauss(EXEMPLOS['Exemplo do material (3×3)'], pivoteamento=False, exato=True, nivel='operacao')
HTML(anim.html(ms=1500))

## 6. Experimentos sugeridos

1. **Exemplo do material, sem pivoteamento, frações exatas** — reproduz exatamente as contas feitas à mão.
2. **Mesmo exemplo com pivoteamento parcial** — na coluna 1 o maior módulo é $|-3|$, então $R_1 \leftrightarrow R_2$; a matriz triangular muda, a solução não.
3. **Pivô nulo (2×2)** — sem pivoteamento a eliminação para (divisão por zero); com pivoteamento troca as linhas e dá $x=1,\ y=2$.
4. **Pivô muito pequeno, em ponto flutuante** — sem pivoteamento o multiplicador ≈ $5	imes10^{13}$ amplifica o arredondamento e $x pprox 0{,}9992$ (erro ≈ $8	imes10^{-4}$); com pivoteamento o erro some.
5. **Hilbert n×n em ponto flutuante** — resíduo minúsculo, mas erro bem maior: resíduo pequeno não garante solução precisa.
6. **SPI / SI** — a eliminação termina com uma linha $0 = 0$ (infinitas soluções) ou $0 = c
eq 0$ (sem solução).
7. **Mude a dimensão n** e sorteie sistemas até 8×8: o número de operações de linha cresce como $n(n-1)/2$.